In [1]:
import numpy as np

In [2]:
import pandas as pd

In [3]:
sales_23 = pd.read_csv('sales_2023.csv')

In [4]:
sales_24 = pd.read_csv('sales_2024.csv')

In [6]:
customer = pd.read_csv('customer_profiles.csv')

In [7]:
sales_23.head()

,Customer_ID,Customer_Name,Region_ID,Category,Q1_Sales,Q2_Sales,Q3_Sales,Q4_Sales
0,CUST-1001,Grace Nelson,R2_South,Clothing,819,903,317,465
1,CUST-1002,Danny Page,R3_East,Office Supplies,808,184,319,327
2,CUST-1003,Helen Hopper,R3_East,Office Supplies,806,876,542,689
3,CUST-1004,Trish Page,R3_East,Home Decor,269,239,944,881
4,CUST-1005,Karen Miller,R4_West,Office Supplies,936,925,783,190


melt() transforms data from WIDE to LONG format. It collapses multiple columns into key-value pairs (un-pivoting).

In [9]:
# Minimal version
df_long = sales_23.melt(
    id_vars=["Customer_ID", "Customer_Name", "Region_ID", "Category"]
)

In [10]:
df_long

,Customer_ID,Customer_Name,Region_ID,Category,variable,value
0,CUST-1001,Grace Nelson,R2_South,Clothing,Q1_Sales,819
1,CUST-1002,Danny Page,R3_East,Office Supplies,Q1_Sales,808
2,CUST-1003,Helen Hopper,R3_East,Office Supplies,Q1_Sales,806
3,CUST-1004,Trish Page,R3_East,Home Decor,Q1_Sales,269
4,CUST-1005,Karen Miller,R4_West,Office Supplies,Q1_Sales,936
...,...,...,...,...,...,...
395,CUST-1096,Frank Castle,R3_East,Clothing,Q4_Sales,378
396,CUST-1097,Alice Malcolm,R3_East,Home Decor,Q4_Sales,727
397,CUST-1098,Frank Castle,R1_North,Furniture,Q4_Sales,224
398,CUST-1099,Charlie Nelson,R4_West,Clothing,Q4_Sales,584


In [19]:
# Explicit version
df_long = sales_23.melt(
    id_vars=["Customer_ID", "Customer_Name", "Region_ID", "Category"],
    value_vars=["Q1_Sales", "Q2_Sales", "Q3_Sales", "Q4_Sales"],  # Optional
    var_name="Quarter",  # Renames 'variable' to 'Quarter'
    value_name="Sales",  # Renames 'value' to 'Sales'
)

In [13]:
df_long

,Customer_ID,Customer_Name,Region_ID,Category,Quarter,Sales
0,CUST-1001,Grace Nelson,R2_South,Clothing,Q1_Sales,819
1,CUST-1002,Danny Page,R3_East,Office Supplies,Q1_Sales,808
2,CUST-1003,Helen Hopper,R3_East,Office Supplies,Q1_Sales,806
3,CUST-1004,Trish Page,R3_East,Home Decor,Q1_Sales,269
4,CUST-1005,Karen Miller,R4_West,Office Supplies,Q1_Sales,936
...,...,...,...,...,...,...
395,CUST-1096,Frank Castle,R3_East,Clothing,Q4_Sales,378
396,CUST-1097,Alice Malcolm,R3_East,Home Decor,Q4_Sales,727
397,CUST-1098,Frank Castle,R1_North,Furniture,Q4_Sales,224
398,CUST-1099,Charlie Nelson,R4_West,Clothing,Q4_Sales,584


pivot() transforms data from LONG to WIDE format. It expands unique values from one column into multiple separate column headers.

In [20]:
# Turns df_long back into a wide dataframe per customer
df_wide = df_long.pivot(index="Customer_ID", columns="Quarter", values="Sales")

df_wide.head()

Quarter,Q1_Sales,Q2_Sales,Q3_Sales,Q4_Sales
Customer_ID,,,,
CUST-1001,819,903,317,465
CUST-1002,808,184,319,327
CUST-1003,806,876,542,689
CUST-1004,269,239,944,881
CUST-1005,936,925,783,190


.pivot() ONLY reshapes unique rows;
.pivot_table() aggregates grouped rows using math (e.g., aggfunc='sum').

In [22]:
# Summarizes total sales by category and quarter with row/column totals
pivot_cat = df_long.pivot_table(
    index="Category",
    columns="Quarter",
    values="Sales",
    aggfunc="sum",
    margins=True,  # Adds "Total" row and column
    margins_name="Total",
)

pivot_cat

Quarter,Q1_Sales,Q2_Sales,Q3_Sales,Q4_Sales,Total
Category,,,,,
Clothing,12314,12869,12585,13901,51669
Electronics,10602,10096,9157,8713,38568
Furniture,11413,13028,10354,11171,45966
Home Decor,13138,9686,10827,11075,44726
Office Supplies,10041,9835,9662,9843,39381
Total,57508,55514,52585,54703,220310


pd.concat(axis=0) stacks DataFrames top-to-bottom by matching column names, which is ideal for combining datasets with identical schemas like monthly or yearly sales logs.

pd.concat(axis=1) stitches DataFrames side-by-side by matching row indices, performing a full outer join by default and filling non-matching index rows with NaN.

pd.concat(axis=1) with .set_index() lets you align side-by-side data using specific column values instead of row positions by first promoting those columns into the row index.

In [30]:
# Vertical stacking: 100 rows + 100 rows = 200 rows total
df_stacked = pd.concat([sales_23, sales_24], axis=0, ignore_index=True)
df_stacked.head()

,Customer_ID,Customer_Name,Region_ID,Category,Q1_Sales,Q2_Sales,Q3_Sales,Q4_Sales
0,CUST-1001,Grace Nelson,R2_South,Clothing,819,903,317,465
1,CUST-1002,Danny Page,R3_East,Office Supplies,808,184,319,327
2,CUST-1003,Helen Hopper,R3_East,Office Supplies,806,876,542,689
3,CUST-1004,Trish Page,R3_East,Home Decor,269,239,944,881
4,CUST-1005,Karen Miller,R4_West,Office Supplies,936,925,783,190


In [33]:
# Stitching horizontally by index (axis=1)
df_horizontal = pd.concat([sales_23, sales_24], axis=1)
df_horizontal.head()

,Customer_ID,Customer_Name,Region_ID,Category,Q1_Sales,Q2_Sales,Q3_Sales,Q4_Sales,Customer_ID,Customer_Name,Region_ID,Category,Q1_Sales,Q2_Sales,Q3_Sales,Q4_Sales
0,CUST-1001,Grace Nelson,R2_South,Clothing,819,903,317,465,CUST-1101,Frank Jones,R1_North,Clothing,509,883,888,546
1,CUST-1002,Danny Page,R3_East,Office Supplies,808,184,319,327,CUST-1102,Frank Wing,R2_South,Home Decor,710,375,393,871
2,CUST-1003,Helen Hopper,R3_East,Office Supplies,806,876,542,689,CUST-1103,Bob Wing,R4_West,Clothing,476,202,640,310
3,CUST-1004,Trish Page,R3_East,Home Decor,269,239,944,881,CUST-1104,Danny Walker,R3_East,Home Decor,918,649,656,165
4,CUST-1005,Karen Miller,R4_West,Office Supplies,936,925,783,190,CUST-1105,Bob Nelson,R3_East,Home Decor,885,655,514,642


`join` ONLY accepts two values: 'outer' (default) or 'inner'.

In [40]:
# Modern Syntax: df1.join(df2, ...)
df_joined = sales_23.join(
    other=sales_24,
    on="Customer_ID",  # Optional: use if sales_23 has Customer_ID as a column instead of an index
    how="left",  # Options: 'left', 'right', 'inner', 'outer'
    lsuffix="_23",  # Suffix for overlapping columns from sales_23
    rsuffix="_24",  # Suffix for overlapping columns from sales_24
)

ValueError: You are trying to merge on object and int64 columns for key 'Customer_ID'. If you wish to proceed you should use pd.concat

In [43]:
# Function call passing two DataFrames
df_result = pd.merge(
    left=sales_23,
    right=sales_24,
    on="Customer_ID",  # Column to join on (must exist in both)
    how="left",  # Join type: 'inner', 'left', 'right', or 'outer'
    suffixes=("_23", "_24"),  # Appended to overlapping column names
)
df_result.head()

,Customer_ID,Customer_Name_23,Region_ID_23,Category_23,Q1_Sales_23,Q2_Sales_23,Q3_Sales_23,Q4_Sales_23,Customer_Name_24,Region_ID_24,Category_24,Q1_Sales_24,Q2_Sales_24,Q3_Sales_24,Q4_Sales_24
0,CUST-1001,Grace Nelson,R2_South,Clothing,819,903,317,465,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CUST-1002,Danny Page,R3_East,Office Supplies,808,184,319,327,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CUST-1003,Helen Hopper,R3_East,Office Supplies,806,876,542,689,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,CUST-1004,Trish Page,R3_East,Home Decor,269,239,944,881,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CUST-1005,Karen Miller,R4_West,Office Supplies,936,925,783,190,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [42]:
# Basic Left Join on Index
df_joined = sales_23.set_index("Customer_ID").join(
    sales_24.set_index("Customer_ID"), lsuffix="_23", rsuffix="_24"
)
df_joined

,Customer_Name_23,Region_ID_23,Category_23,Q1_Sales_23,Q2_Sales_23,Q3_Sales_23,Q4_Sales_23,Customer_Name_24,Region_ID_24,Category_24,Q1_Sales_24,Q2_Sales_24,Q3_Sales_24,Q4_Sales_24
Customer_ID,,,,,,,,,,,,,,
CUST-1001,Grace Nelson,R2_South,Clothing,819,903,317,465,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CUST-1002,Danny Page,R3_East,Office Supplies,808,184,319,327,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CUST-1003,Helen Hopper,R3_East,Office Supplies,806,876,542,689,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CUST-1004,Trish Page,R3_East,Home Decor,269,239,944,881,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CUST-1005,Karen Miller,R4_West,Office Supplies,936,925,783,190,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CUST-1096,Frank Castle,R3_East,Clothing,161,402,497,378,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CUST-1097,Alice Malcolm,R3_East,Home Decor,756,379,825,727,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CUST-1098,Frank Castle,R1_North,Furniture,451,668,239,224,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
# Modern Syntax: pd.merge(left, right, ...)
df_merged = pd.merge(
    left=sales_23,
    right=sales_24,
    on="Customer_ID",  # The matching column name in both tables
    how="left",  # Options: 'left', 'right', 'inner', 'outer', 'cross'
    suffixes=("_23", "_24"),  # Suffixes added to duplicate column names
)
df_merged

,Customer_ID,Customer_Name_23,Region_ID_23,Category_23,Q1_Sales_23,Q2_Sales_23,Q3_Sales_23,Q4_Sales_23,Customer_Name_24,Region_ID_24,Category_24,Q1_Sales_24,Q2_Sales_24,Q3_Sales_24,Q4_Sales_24
0,CUST-1001,Grace Nelson,R2_South,Clothing,819,903,317,465,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CUST-1002,Danny Page,R3_East,Office Supplies,808,184,319,327,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CUST-1003,Helen Hopper,R3_East,Office Supplies,806,876,542,689,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,CUST-1004,Trish Page,R3_East,Home Decor,269,239,944,881,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CUST-1005,Karen Miller,R4_West,Office Supplies,936,925,783,190,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,CUST-1096,Frank Castle,R3_East,Clothing,161,402,497,378,NaN,NaN,NaN,NaN,NaN,NaN,NaN
96,CUST-1097,Alice Malcolm,R3_East,Home Decor,756,379,825,727,NaN,NaN,NaN,NaN,NaN,NaN,NaN
97,CUST-1098,Frank Castle,R1_North,Furniture,451,668,239,224,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98,CUST-1099,Charlie Nelson,R4_West,Clothing,402,323,797,584,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [47]:
df_merged = (
    sales_23.merge(sales_24, on="Customer_ID", how="left")
    .merge(customer, on="Customer_ID", how="left")
)
df_merged

,Customer_ID,Customer_Name_x,Region_ID_x,Category_x,Q1_Sales_x,Q2_Sales_x,Q3_Sales_x,Q4_Sales_x,Customer_Name_y,Region_ID_y,Category_y,Q1_Sales_y,Q2_Sales_y,Q3_Sales_y,Q4_Sales_y,Customer_Segment,Loyalty_Tier,Credit_Score,Account_Created_Year
0,CUST-1001,Grace Nelson,R2_South,Clothing,819,903,317,465,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Consumer,Silver,793,2020
1,CUST-1002,Danny Page,R3_East,Office Supplies,808,184,319,327,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Small Business,Gold,696,2021
2,CUST-1003,Helen Hopper,R3_East,Office Supplies,806,876,542,689,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Home Office,Bronze,651,2020
3,CUST-1004,Trish Page,R3_East,Home Decor,269,239,944,881,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Small Business,Gold,643,2022
4,CUST-1005,Karen Miller,R4_West,Office Supplies,936,925,783,190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Home Office,Bronze,784,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,CUST-1096,Frank Castle,R3_East,Clothing,161,402,497,378,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Home Office,Silver,675,2021
96,CUST-1097,Alice Malcolm,R3_East,Home Decor,756,379,825,727,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Consumer,Bronze,684,2019
97,CUST-1098,Frank Castle,R1_North,Furniture,451,668,239,224,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Home Office,Silver,829,2021
98,CUST-1099,Charlie Nelson,R4_West,Clothing,402,323,797,584,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Consumer,Bronze,696,2019


In [48]:
# 1. First merge: sales_23 + sales_24
sales_combined = pd.merge(
    sales_23,
    sales_24,
    on="Customer_ID",
    how="left",
    suffixes=("_23", "_24"),
)

# 2. Second merge: (sales_23 + sales_24 result) + customer_profiles
final_df = pd.merge(
    sales_combined,
    customer,
    on="Customer_ID",
    how="left",
)

# Check the result
final_df.head()

,Customer_ID,Customer_Name_23,Region_ID_23,Category_23,Q1_Sales_23,Q2_Sales_23,Q3_Sales_23,Q4_Sales_23,Customer_Name_24,Region_ID_24,Category_24,Q1_Sales_24,Q2_Sales_24,Q3_Sales_24,Q4_Sales_24,Customer_Segment,Loyalty_Tier,Credit_Score,Account_Created_Year
0,CUST-1001,Grace Nelson,R2_South,Clothing,819,903,317,465,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Consumer,Silver,793,2020
1,CUST-1002,Danny Page,R3_East,Office Supplies,808,184,319,327,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Small Business,Gold,696,2021
2,CUST-1003,Helen Hopper,R3_East,Office Supplies,806,876,542,689,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Home Office,Bronze,651,2020
3,CUST-1004,Trish Page,R3_East,Home Decor,269,239,944,881,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Small Business,Gold,643,2022
4,CUST-1005,Karen Miller,R4_West,Office Supplies,936,925,783,190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Home Office,Bronze,784,2020
